# Meningioma Challenge 2023 Analysis

This notebook will reflect an exhaustive Data Analysis about the Meningiomas presents in the BraTS2023 MEN challenge. 

## Questions to answer

- Most effective modality for detecting meningiomas
- Size distribution of meningiomas ✅
- Distribution of slices containing meningiomas ✅
- Number of slices with more than 0.01 tumor presence ✅
- Number of slices with no tumor presence ✅
- ...

In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
import altair as alt

#### Provisional, how to read h5 files

In [ ]:
# Read an slice from the nifti file
path = r'D:\data\BraTS2023-MEN-adapted\train\t1c\BraTS-MEN-00004-000_49_t1c.h5'

# Read the h5 file
with h5py.File(path, 'r') as f:
    data = f['data'][:]

# Plot the slice
plt.imshow(data, cmap='gray')
plt.show()

print(f"Max value: {np.max(data)}")
print(f"Min value: {np.min(data)}")

## Data Example

In [ ]:
# dataset path
path_brats = r'D:/data//BraTS2023-MEN/BraTS-MEN-00131-000'
fig, axes = plt.subplots(1, len(os.listdir(path_brats)), figsize=(15, 5))

for i, modality in enumerate(os.listdir(path_brats)):
    file = os.path.join(path_brats, modality)
    
    img = nib.load(file)
    data = img.get_fdata()

    slice = data[:,:,80]

    axes[i].imshow(slice, cmap='gray')
    axes[i].set_title(modality.split('-')[-1].split('.')[0])
    axes[i].axis('off')

plt.tight_layout()
plt.show()


It seems that t1c is the best for seeing the actual tunour and t2f the best for seeing the 'water' of the tumour

## Dataset preprocessing

Create a dataframe where we analyse each slice of each patient and determine the amount of tumour of each one (around 15 min to process the thousand brains)

In [ ]:
# dataset path
path_brats = r'D:/data/BraTS2023-MEN'

# For each patient in the dataset, see each slice and save in a dataset the amount of pixels that are not 0 in 'seg' and the amount of pixels that are not 0 in 't1ce'

results = []

for patient in os.listdir(path_brats):
    path_patient = os.path.join(path_brats, patient)
    for modality in os.listdir(path_patient):
        if 'seg' in modality:

            file = os.path.join(path_patient, modality)
            
            img = nib.load(file)
            data = img.get_fdata()

            for slice_idx in range(data.shape[2]):
                slice = data[:,:,slice_idx]
                non_zero_pixels = np.count_nonzero(slice)

                # Append the results for each slice
                results.append({
                    'patient': patient,
                    'modality': modality.split('-')[-1].split('.')[0],
                    'slice': slice_idx,
                    'non_zero_pixels': non_zero_pixels,

                })

        # Convert the results list to a DataFrame
        df_results = pd.DataFrame(results)

In [ ]:
# Save the DataFrame to a CSV file
df_results.to_csv('D:/data/meningioma_slices.csv', index=False)


In [ ]:
# Read the CSV file
path = r'C:\Users\gerar\Documents\TFG\visual_transformers_anomaly_segmentation\data_analysis\meningioma_slices.csv'
df_results_r = pd.read_csv(path)

## Dataset Composition

### 1. Number of slices with no tumor presence

For each row with modality `seg` count if they have tumour or not

In [ ]:
# Count the slices with non-zero pixels in 'seg'
dist_sizes = df_results_r[df_results_r['modality'] == 'seg']['non_zero_pixels'].value_counts()

In [ ]:
dist_sizes[0]

The number of slices that they have no tumour is `115.406`

In [ ]:
# Count the total slices in 'seg'
total_slices = df_results_r[df_results_r['modality'] == 'seg'].shape[0]
total_slices

In [ ]:
dist_sizes[0] / total_slices

The total number of slices is `155.000`, this means that the 74.45% doesn't present tumour and could be selected for trainning.

In [ ]:
total_slices - dist_sizes[0]

The number of slices with tumour is `39.594`

### 2. Number of slices with more than 0.01 tumor presence

For each row in `seg` look the percentatge of tumour

In [ ]:
# Percentage of tumour area in each slice of the seg modality
df_results_r.loc[df_results_r['modality'] == 'seg', 'percentage'] = df_results_r['non_zero_pixels'] / (240 * 240) * 100


In [ ]:
# Count the amount of slices with a percentage of tumour area greater than 0.01% (more than 6 pixels of tumour)
df_results_r[df_results_r['percentage'] > 0.01].shape[0]


In [ ]:
39040/155000*100

In [ ]:
# Save the DataFrame to a CSV file
df_results_r.to_csv('meningioma_slices_p.csv', index=False)

The amount of slices with less than 0.01% tumour is 39.594-39.040 = 546, so we will discard this ones.

### 3. Patient Demographics

We will study briefly the distribution of the patient's age in the dataset and their sex. The metadata can be obtained in the `Meningioma supplementary clinical data and imaging parameters for training and validation sets.xlx`. The metadata is not available for the test data, so this information will not be really important. 

In [ ]:
# Read the xlsx file
xlsx_path = r'D:\data\BraTS2023-MEN\Meningioma supplementary clinical data and imaging parameters for training and validation sets.xlsx'
df_xlsx = pd.read_excel(xlsx_path)

df_xlsx.head()

In [ ]:
# Create a histogram of the age distribution
age_hist = alt.Chart(df_xlsx).mark_bar(color='#F4A582').encode(
    alt.X('Age:Q', bin=alt.Bin(maxbins=30), title='Age'),
    alt.Y('count()', title='Frequency')
).properties(
    title='Age Distribution of Patients'
)

# Compute the mean and the std
mean_age = df_xlsx['Age'].mean()
std_age = df_xlsx['Age'].std()

# Display the histogram
age_hist.display()

mean_age, std_age

In [ ]:
# Calculate the percentage of each sex
sex_counts = df_xlsx['Sex'].value_counts(normalize=True).reset_index()
sex_counts.columns = ['Sex', 'Percentage']
sex_counts['Percentage'] *= 100

# Create a pie chart
sex_chart = alt.Chart(sex_counts).mark_arc().encode(
    theta=alt.Theta(field='Percentage', type='quantitative', stack=True),
    color=alt.Color(field='Sex', type='nominal', scale=alt.Scale(domain=['Female', 'Male'], range=['#76C7C5', '#E9B872']))
).properties(
    title='Percentage of Sex of the Patients'
)

text = sex_chart.mark_text(radius=115, fill="black").encode(alt.Text(field="Percentage", type="quantitative", format=",.1f"))

# Display the chart
(sex_chart + text).display()

## Statistical MRI Analysis

### 4. Distribution of slices containing meningiomas

The idea is to compute the percentatge of each slice contains a meningioma, the average nonzero pixels and if it is nonzero the average non pixels

In [ ]:
# Prepare the data for Altair
df_grouped_total = df_results_r[df_results_r['modality'] == 'seg'].groupby('slice')['percentage'].count().reset_index()
df_grouped = df_results_r[(df_results_r['modality'] == 'seg') & (df_results_r['percentage'] > 0)].groupby('slice')['percentage'].count().reset_index()

df_grouped_total.columns = ['slice', 'total_slices']
df_grouped.columns = ['slice', 'tumour_slices']

df_merged = pd.merge(df_grouped_total, df_grouped, on='slice')
df_merged['percentage'] = (df_merged['tumour_slices'] / df_merged['total_slices']) * 100

# Create the Altair chart
chart = alt.Chart(df_merged).mark_line(color='#F4A582').encode(
    x=alt.X('slice:Q', title='Slice'),
    y=alt.Y('percentage:Q', title='Percentage of Slices with Tumour (%)')
).properties(
    title='Percentage of Slices with Tumour by Slice'
)

# Display the chart
chart.display()


In [ ]:
# Prepare the data for Altair
df_grouped_total = df_results_r[df_results_r['modality'] == 'seg'].groupby('slice')['percentage'].count().reset_index()
df_grouped = df_results_r[(df_results_r['modality'] == 'seg') & (df_results_r['percentage'] > 0)].groupby('slice')['percentage'].mean().reset_index()

df_grouped_total.columns = ['slice', 'total_slices']
df_grouped.columns = ['slice', 'mean_percentage']

# Merge the dataframes
df_merged = pd.merge(df_grouped_total, df_grouped, on='slice')
df_merged['key'] = 'Tumour'

# Create the Altair chart
mean_percentage_chart = alt.Chart(df_merged).mark_line().encode(
    x=alt.X('slice:Q', title='Slice'),
    y=alt.Y('mean_percentage:Q', title='Mean Percentage of Tumour Area')
)

# Prepare the data for Altair
df_grouped = df_results_r[df_results_r['modality'] == 'seg'].groupby('slice')['percentage'].mean().reset_index()
df_grouped['key'] = 'All'

# Create the Altair chart
chart = alt.Chart(df_grouped).mark_line().encode(
    x=alt.X('slice:Q', title='Slice'),
    y=alt.Y('percentage:Q', title='Mean Percentage of Tumour Area')
)

# Display the chart
(mean_percentage_chart + chart).encode(
    color=alt.Color('key:N', legend=alt.Legend(title="Legend", orient="top-right"),
                    scale=alt.Scale(domain=['Tumour', 'All'], range=['#F4D03F', '#F4A582']))
).properties(
    title='Distribution of Tumour Area by Slice'
).display()

### 5. Statistics for each modality (mean, std, min, max, skewness, kurtosis)

In [ ]:
def compute_dataset_statistics(data_dir, modalities=['seg', 't1n', 't1c', 't2w', 't2f']):
    """
    Computes min, max, mean, standard deviation, skewness, and kurtosis for each MRI modality.
    
    Parameters:
    - data_dir (str): Path to the dataset directory.
    - modalities (list): List of modality names.

    Returns:
    - dict: A dictionary containing statistics for each modality.
    """
    # Initialize statistics containers
    min_value = np.full(len(modalities), np.inf)
    max_value = np.full(len(modalities), -np.inf)
    total_sum = np.zeros(len(modalities))
    total_sum_sq = np.zeros(len(modalities))
    total_voxels = np.zeros(len(modalities))

    for i, modality in enumerate(modalities):
        path_modality = os.path.join(data_dir, modality)
        
        for patient in os.listdir(path_modality):
            image_path = os.path.join(path_modality, patient)
            
            # Load the image based on file type
            if image_path.endswith(('.nii', '.nii.gz')):
                img = nib.load(image_path)
                image_data = img.get_fdata()
            elif image_path.endswith('.h5'):
                with h5py.File(image_path, 'r') as f:
                    image_data = f['data'][:]
            else:
                continue  # Skip unsupported file formats

            # Update min and max values
            min_value[i] = min(min_value[i], np.min(image_data))
            max_value[i] = max(max_value[i], np.max(image_data))

            # Update sum and sum of squares for mean and std calculations
            total_sum[i] += np.sum(image_data)
            total_sum_sq[i] += np.sum(image_data ** 2)
            total_voxels[i] += np.prod(image_data.shape)

    # Compute statistics
    mean = total_sum / total_voxels
    variance = (total_sum_sq / total_voxels) - (mean ** 2)
    std_dev = np.sqrt(variance)

    # Store results in a dictionary
    stats = {
        'Min': min_value,
        'Max': max_value,
        'Mean': mean,
        'StdDev': std_dev,
    }

    return stats

In [ ]:
# Define train dataset path
path_brats = r'\Users\gerar\Documents\TFG\visual_transformers_anomaly_segmentation\data\BraTS2023-MEN-adapted\train'

# Compute statistics
dataset_stats = compute_dataset_statistics(path_brats)

# Print results
for stat_name, values in dataset_stats.items():
    print(f"{stat_name}: {values}")

In [ ]:
# Define test dataset path
path_brats = r'\Users\gerar\Documents\TFG\visual_transformers_anomaly_segmentation\data\BraTS2023-MEN-adapted\test'

# Compute statistics
dataset_stats = compute_dataset_statistics(path_brats, modalities=['t1n', 't1c', 't2w', 't2f'])

# Print results
for stat_name, values in dataset_stats.items():
    print(f"{stat_name}: {values}")

In [ ]:
train_stats = pd.DataFrame({
    'Min': [-0.49562917, -0.60397325, -0.80070544, -0.48707063],
    'Max': [22.0747628, 19.10432155, 10.89006929, 13.84331785],
    'Mean': [-0.04931307, -0.05128799, -0.04654453, -0.05106495],
    'StdDev': [0.9477391, 0.93575664, 0.94779331, 0.93920239]
})
train_stats['Dataset'] = 'Train'

test_stats = pd.DataFrame({
    'Min': [-0.49081902, -0.5884039, -0.47749385, -0.48178286],
    'Max': [10.02755707, 18.69862506, 10.79106811, 13.8839156],
    'Mean': [0.16379892, 0.16801783, 0.1556128, 0.16973017],
    'StdDev': [1.14099351, 1.1672222, 1.14722018, 1.16579419]
})
test_stats['Dataset'] = 'Test'

# Combine the data
combined_stats = pd.concat([train_stats, test_stats], ignore_index=True)
combined_stats['Modality'] = ['t1n', 't1c', 't2w', 't2f'] * 2

# Melt the dataframe for Altair
melted_stats = combined_stats.melt(id_vars=['Dataset', 'Modality'], var_name='Statistic', value_name='Value')

# Create the Altair chart
chart = alt.Chart(melted_stats).mark_bar().encode(
    x=alt.X('Modality:N', title='Modality'),
    y=alt.Y('Value:Q', title='Value', axis=alt.Axis(grid=False)),
    color=alt.Color('Dataset:N', title='Dataset', scale=alt.Scale(domain=['Train', 'Test'], range=['#480454', '#e8e41c'])),
    column=alt.Column('Statistic:N', title='Statistic', sort=['Max', 'Min', 'Mean', 'StdDev']),
    xOffset=alt.XOffset('Dataset:N')
).properties(
    title='Comparison of Train and Test Data Statistics'
).resolve_scale(
    y='independent'
)

chart


### 6. Study the distribution of the intensity for each modality

In [ ]:
from tqdm import tqdm

sys.path.append(os.path.abspath('../code/methods/datasets'))

from DatasetBlosc2 import DatasetBlosc2


# Function to plot the histogram of pixel intensities for each slice
def compute_intensity_distribution(data_dir):

    dataset = DatasetBlosc2(data_dir, identifiers=None)
    patients = dataset.identifiers
    results = np.zeros((4, 240*240))
    slices = 0

    for patient in tqdm(patients, desc='Processing patients', leave=False):
        data, _ = dataset.load_case(patient)

        data = np.array(data, dtype=float)

        for slice_idx in range(data.shape[-1]):
            slice_data = data[:,:,:,slice_idx]
            results += slice_data.reshape(4, 240*240)

        slices += data.shape[-1]

    results /= slices
    
    return results

In [ ]:
# Dataset path
train_path_brats = r'D:\data\BraTS2023-MEN-adapted-Blosc2-zscore_brain\train'
test_stats_brats = r'D:\data\BraTS2023-MEN-adapted-Blosc2-zscore_brain\test'

# Plot the intensity distribution
train_i = compute_intensity_distribution(train_path_brats)
test_i = compute_intensity_distribution(test_stats_brats)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for modality, train, test, ax in zip(['t1n', 't1c', 't2w', 't2f'], train_i, test_i, axes[:4]):
    # Plot the intensity distribution
    ax.hist(train, bins=50, color='#480454', alpha=0.7)
    ax.hist(test, bins=50, color='#e8e41c', alpha=0.7)
    ax.set_title(f'Intensity Distribution for {modality}')
    ax.set_xlabel('Intensity')
    ax.set_ylabel('Frequency')

# Create a single legend
fig.legend(['Train', 'Test'], loc='upper right')

plt.tight_layout()
plt.show()

## Tumor Specific Analysis 

### 7. Size distribution of meningiomas

In [ ]:
# Group by patient and calculate the mean percentage of tumor area
patient_tumor_percentage = df_results_r[df_results_r['modality'] == 'seg'].groupby('patient')['percentage'].mean().reset_index()

# Create a histogram of the percentage of tumor area per patient
percentage_hist = alt.Chart(patient_tumor_percentage).mark_bar(color='#F4A582').encode(
    alt.X('percentage:Q', bin=alt.Bin(maxbins=24), title='Percentage of tumour area'),
    alt.Y('count()', title='Number of patients')
).properties(
    title='Distribution of the percentage of tumour area per patient',
    width=600,
    height=400
)

# Display the histogram
percentage_hist.display()

# Compute mean and std
mean_percentage = patient_tumor_percentage['percentage'].mean()
std_percentage = patient_tumor_percentage['percentage'].std()

mean_percentage, std_percentage


In [ ]:
# histogram from 0 to 0.20
# Group by patient and calculate the mean percentage of tumor area
patient_tumor_percentage = df_results_r[df_results_r['modality'] == 'seg'].groupby('patient')['percentage'].mean().reset_index()

# Filter the data to include only percentages from 0 to 0.2
filtered_patient_tumor_percentage = patient_tumor_percentage[(patient_tumor_percentage['percentage'] >= 0) & (patient_tumor_percentage['percentage'] <= 0.2)]

# Create a histogram of the percentage of tumor area per patient
percentage_hist = alt.Chart(filtered_patient_tumor_percentage).mark_bar(color='#F4A582').encode(
    alt.X('percentage:Q', bin=alt.Bin(maxbins=128), title='Percentage of tumour area'),
    alt.Y('count()', title='Number of patients')
).properties(
    title='Distribution of the percentage of tumour area per patient (0 to 0.2)',
    width=600,
    height=400
)

# Display the histogram
percentage_hist.display()

# Compute mean and std
mean_percentage = filtered_patient_tumor_percentage['percentage'].mean()
std_percentage = filtered_patient_tumor_percentage['percentage'].std()

mean_percentage, std_percentage

### 8. Number of Slices Containing Tumor per Patient

In [ ]:
# Group by patient and count the number of slices with non-zero pixels in the 'seg' modality
slices_with_tumor_per_patient = df_results_r[(df_results_r['modality'] == 'seg') & (df_results_r['non_zero_pixels'] > 0)].groupby('patient').size().reset_index(name='slices_with_tumor')

# Create a histogram of the number of slices with tumor per patient
slices_hist = alt.Chart(slices_with_tumor_per_patient).mark_bar(color='#F4A582').encode(
    alt.X('slices_with_tumor:Q', bin=alt.Bin(maxbins=20), title='Number of slices with tumor'),
    alt.Y('count()', title='Number of patients'),
    tooltip=['count()']
).properties(
    title='Distribution of the number of slices with tumor per patient',
    width=600,
    height=400
)

# Display the histogram
slices_hist.display()

# Compute the mean and standard deviation of the number of slices with tumor per patient
mean_slices_with_tumor = slices_with_tumor_per_patient['slices_with_tumor'].mean()
std_slices_with_tumor = slices_with_tumor_per_patient['slices_with_tumor'].std()

mean_slices_with_tumor, std_slices_with_tumor

### 9. Tumor Localization

In [ ]:
def plot_mean_segmentation_mask(test_seg_brats, tumor_type='all'):
    slices = np.zeros((240, 240))

    for patient in os.listdir(test_seg_brats):
        image_path = os.path.join(test_seg_brats, patient)
        with h5py.File(image_path, 'r') as f:
            data = f['data'][:]

        if tumor_type != 'all':
            data[data != tumor_type ] = 0
        slices += data

    slices /= len(os.listdir(test_seg_brats))

    # Plot the segmentation mask
    plt.imshow(slices, cmap='viridis')
    plt.colorbar()
    plt.title('Mean Segmentation Mask' if tumor_type == 'all' else f'Mean Segmentation Mask for Tumor Type {tumor_type}')
    plt.axis('off')
    plt.show()      

In [ ]:
# Dataset path
test_seg_brats = r'\Users\gerar\Documents\TFG\visual_transformers_anomaly_segmentation\data\BraTS2023-MEN-adapted\test\seg'

# 0 - background
# 1 - non-enhancing tumor
# 2 - edema
# 3 - enhancing tumor

# Plot the mean segmentation mask for all tumor types
plot_mean_segmentation_mask(test_seg_brats)

In [ ]:
# Plot the mean segmentation mask for non-enhancing tumor
plot_mean_segmentation_mask(test_seg_brats, tumor_type=1)

In [ ]:
# Plot the mean segmentation mask for edema
plot_mean_segmentation_mask(test_seg_brats, tumor_type=2)

In [ ]:
# Plot the mean segmentation mask for enhancing tumor
plot_mean_segmentation_mask(test_seg_brats, tumor_type=3)

### 10. Distribution of the intensity of the tumor areas for each modality 

In [ ]:
from tqdm import tqdm
import sys
import os

sys.path.append(os.path.abspath('../code/methods/datasets'))

from DatasetBlosc2 import DatasetBlosc2


def compute_intensity_tumor_distribution(data_dir):

    dataset = DatasetBlosc2(data_dir, identifiers=None)
    patients = dataset.identifiers
    results = []

    for patient in tqdm(patients, desc='Processing patients', leave=False):
        data, seg = dataset.load_case(patient)

        seg = np.array(seg, dtype=int)
        data = np.array(data, dtype=float)

        masks = {
            'All Tumor': seg > 0,
            'Enhancing Tumor': seg == 3,
            'Non-Enhancing Tumor': seg == 1,
            'Edema': seg == 2
        }

        means = {}
        for tumor_type, mask in masks.items():
            masked_data = data * mask
            mean_intensity = masked_data.sum(axis=(1, 2, 3)) / mask.sum() if mask.sum() > 0 else np.zeros(data.shape[0])
            means[tumor_type] = mean_intensity

        modalities = ['t1c', 't1n', 't2w', 't2f']

        for i in range(data.shape[0]):
            for tumor_type, mean in means.items():
                results.append({
                    'patient': patient,
                    'Tumor Type': tumor_type,
                    'Modality': modalities[i],
                    'Mean Intensity': mean[i]
                })

                print(results[-1])


    return results

In [ ]:
# Dataset path
test_stats_brats = r'D:\data\BraTS2023-MEN-adapted-Blosc2-zscore_brain\test'
tumor_intensity = compute_intensity_tumor_distribution(test_stats_brats)

In [ ]:
# save results
tumor_intensity_df = pd.DataFrame(tumor_intensity)
tumor_intensity_df.to_csv('tumor_intensity.csv', index=False)

In [ ]:
# Define the custom colors
colors = {
    'All Tumor': '#e8e41c',          # Amarillo pastel
    'Enhancing Tumor': '#81D4FA',     # Azul celeste pastel
    'Edema': '#D4E157',               # Verde lima pastel
    'Non-Enhancing Tumor': '#FF8A80'  # Rojo salmón pastel
}

df_tumor_intensity = pd.DataFrame(tumor_intensity)

df_mean_intensity = df_tumor_intensity.groupby(['Modality', 'Tumor Type'])['Mean Intensity'].mean().reset_index()

alt.data_transformers.disable_max_rows()

In [ ]:
boxplot = alt.Chart(df_tumor_intensity).mark_boxplot(extent="min-max").encode(
    x=alt.X('Modality:N', title='Modality', axis=alt.Axis(labelAngle=0)),  # Adjust axis and titles
    y=alt.Y('Mean Intensity:Q', title='Mean Intensity'),
    xOffset=alt.XOffset('Tumor Type:N', title='Tumor Type'),
    color=alt.Color('Tumor Type:N', 
                    scale=alt.Scale(domain=list(colors.keys()), range=list(colors.values())), 
                    title='Tumor Type'),
    tooltip=['Modality:N', 'Tumor Type:N', 'Mean Intensity:Q']  # Add tooltips for better interaction
).properties(
    title='Distribution of Mean Intensity by Tumor Type and Modality',
    width=600,  # You can adjust width and height for better clarity
    height=400
).configure_view(
    stroke=None  # Remove border around the plot for a cleaner look
)

boxplot.display()

In [ ]:
intensity_chart = alt.Chart(df_mean_intensity).mark_bar().encode(
    x=alt.X('Modality', title='Modality'),
    y=alt.Y('Mean Intensity', title='Mean Intensity'),
    xOffset=alt.XOffset('Tumor Type'),
    color=alt.Color('Tumor Type', title='Tumor Type', scale=alt.Scale(domain=list(colors.keys()), range=list(colors.values())))
).properties(
    title='Mean Intensity of Tumor for each Modality and Tumor Type'
)

intensity_chart.display()

## Intensity Normalization Analysis:

#### Image Intensity Reflects Tissue Type

In [ ]:
from skimage.exposure import match_histograms

def plot_slice_histogram(paths, slice_idx='None', normalization='None'):
    # Plot the slice and histogram as subplots
    fig, ax2 = plt.subplots(1, 1, figsize=(12, 6))

    # Plot the histogram
    for path in paths:
        if slice_idx == 'None':
            with h5py.File(path, 'r') as f:
                data = f['data'][:]
        else:
            img = nib.load(path)
            data = img.get_fdata()
            data = data[:, :, slice_idx]

        if normalization == 'zscore':
            data = (data - data.mean()) / data.std()
        elif normalization == 'zscore_brain':
            data[data > 0] = (data[data > 0] - data[data > 0].mean()) / data[data > 0].std()
        elif normalization == 'minmax':
            data = (data - data.min()) / (data.max() - data.min())

        min_val = np.min(data)
        print(f"Min value: {min_val}")
        ax2.hist(data[data > min_val], bins=100, alpha=0.7, histtype='step', linewidth=2, label=os.path.basename(path))


    ax2.set_title('Intensity Distribution')
    ax2.set_xlabel('Intensity')
    ax2.set_ylabel('Frequency')
    ax2.legend()

    plt.tight_layout()
    plt.show()


We aplied a patient normalization to have a mean of 0 and a std of 1, based on the mean and std of each patient. 

##### Before Normalization

In [ ]:
volume_path =['D:/data/BraTS2023-MEN/BraTS-MEN-00004-000/BraTS-MEN-00004-000-t1c.nii.gz', 
              'D:/data/BraTS2023-MEN/BraTS-MEN-00008-000/BraTS-MEN-00008-000-t1c.nii.gz',
              'D:/data/BraTS2023-MEN/BraTS-MEN-00010-000/BraTS-MEN-00010-000-t1c.nii.gz', 
              #'D:/data/BraTS2023-MEN/BraTS-MEN-00016-000/BraTS-MEN-00016-000-t1c.nii.gz',
               ]

plot_slice_histogram(volume_path, slice_idx=80)

##### After Normalization

### Distribution of dice score depending on the type of tumour

In [ ]:
# For the test dataset, select the slices that contain each one of the categories:
#   - Enhanced Slices
#   - Non-enhanced Slices
#   - Edema Slices
#   - Enhanced + Non-enhanced Slices
#   - Enhanced + Edema Slices
#   - Non-enhanced + Edema Slices
#   - Enhanced + Non-enhanced + Edema Slices

from tqdm import tqdm
import numpy as np
import sys
import os

sys.path.append(os.path.abspath('../code/methods/datasets'))

from DatasetBlosc2 import DatasetBlosc2


test_dataset_path = "D:/data/BraTS2023-MEN-adapted-Blosc2/test"

test_dataset = DatasetBlosc2(test_dataset_path, identifiers=None)
patients = test_dataset.identifiers

# Initialize a list to store the results
tumor_slices = []

index_i = 0

for patient in tqdm(patients, desc='Processing patients', leave=False):
    data, seg = test_dataset.load_case(patient)

    seg = np.array(seg, dtype=int)

    for slice_idx in range(seg.shape[-1]):
        slice_seg = seg[:, :, slice_idx]

        # Determine the type of tumor in the slice
        tumor_type = []
        if np.any(slice_seg == 3):
            tumor_type.append('Enhancing Tumor')
        if np.any(slice_seg == 1):
            tumor_type.append('Non-Enhancing Tumor')
        if np.any(slice_seg == 2):
            tumor_type.append('Edema')

        # Combine tumor types into a single string
        tumor_type_str = ' + '.join(tumor_type) if tumor_type else 'No Tumor'

        # Append the result to the list
        tumor_slices.append({
            'Volume': patient,
            'Slice': slice_idx,
            'Tumor Type': tumor_type_str
        })
    
    index_i += 1
    if index_i == 10:
        break

# Create a DataFrame from the results
df_tumor_slices = pd.DataFrame(tumor_slices)

# Display the first few rows of the DataFrame
df_tumor_slices.head()

# Save the DataFrame to a CSV file
df_tumor_slices.to_csv('type_tumor_slices_2.csv', index=False)

In [ ]:
# read data metrics
import pandas as pd
import altair as alt

metrics_path = r"C:\Users\gerar\Documents\TFG\visual_transformers_anomaly_segmentation\code\evaluation\results_metrics.csv"
df_metrics = pd.read_csv(metrics_path)

tumour_type_path = r"C:\Users\gerar\Documents\TFG\visual_transformers_anomaly_segmentation\data_analysis\type_tumor_slices.csv"
df_tumour_type = pd.read_csv(tumour_type_path)

In [ ]:
# We will plot the histogram representing the distribution of the WT Dice Score values, but for each column we will have multiple stacked bars
# representing the different kind of slices that we have in the dataset.

# The structure of the df is:
# df_tumour_type:
#   - Patient: the patient ID
#   - Slice: the slice number
#   - Tumor Type: the type of tumor in the slice (e.g., "Enhancing Tumor", "Non-Enhancing Tumor", "Edema")
# df_metrics:
#   - Patient: the patient ID
#   - Slice: the slice number
#   - Metric: the metric name (e.g., "dice")
#   - Value: the value of the metric
# Merge the two DataFrames on 'Volume' and 'Slice'
alt.data_transformers.disable_max_rows()

df_merged = pd.merge(df_tumour_type, df_metrics, on=['Volume', 'Slice'])

# Filter the DataFrame to include only the WT Dice Score values
df_dice = df_merged[df_merged['Metric'] == 'dice'].copy()

# Extract the third value of the 'Value' column and filter values greater than zero
df_dice['Value'] = df_dice['Value'].apply(lambda x: eval(x)[2] if isinstance(x, str) else x[2])
#df_dice = df_dice[df_dice['Value'] > 0]

# Create a selection for the tumor type
tumor_type_selection = alt.selection_point(
    fields=['Tumor Type'],
    bind=alt.binding_select(options=list(df_dice['Tumor Type'].unique()), name='Select Tumor Type: '),
)

# Create a histogram of the WT Dice Score values, with bars filtered by the selected tumor type
hist = alt.Chart(df_dice).mark_bar().encode(
    x=alt.X('Value:Q', bin=alt.Bin(maxbins=50), title='WT Dice Score', scale=alt.Scale(domain=[0, 1])),
    y=alt.Y('count()', title='Frequency'),
    color=alt.Color('Tumor Type:N', title='Tumor Type', scale=alt.Scale(
        domain=['Enhancing Tumor', 'Non-Enhancing Tumor', 'Edema', 
                'Enhancing Tumor + Edema', 'Enhancing Tumor + Non-Enhancing Tumor', 
                'Non-Enhancing Tumor + Edema', 
                'Enhancing Tumor + Non-Enhancing Tumor + Edema'],
        range=['#81D4FA', '#D4E157', '#FF8A80', '#B39DDB', '#00897B', '#FFB74D', '#BDBDBD']
    )),
    tooltip=['count()']
).add_params(
    tumor_type_selection
).transform_filter(
    tumor_type_selection
).properties(
    title='Distribution of WT Dice Score by Tumor Type',
    width=600,
    height=400
)

# Display the histogram
hist.display()

#### Data visualization datasets DA

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../code/methods/datasets'))

from DatasetBlosc2 import DatasetBlosc2
import matplotlib.pyplot as plt
import numpy as np

path = "D:/data/BraTS2024-GoAT-adapted-Blosc2/train"
dataset = DatasetBlosc2(path, identifiers=None)
patients = dataset.identifiers

In [ ]:
# print first volume
data, seg = dataset.load_case(patients[351])
data = np.array(data, dtype=float)
if seg is not None:
    seg = np.array(seg, dtype=int)

In [ ]:
# plot first volume
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(data[3, :, :, 80], cmap='gray')
# plt.title('Data Volume')
plt.axis('off')
if seg is not None:
    plt.subplot(1, 2, 2)
    plt.imshow(seg[:, :, 80], cmap='gray')
    plt.title('Segmentation Volume')
    plt.axis('off')
    plt.show()


In [ ]:
from einops import rearrange

img = np.expand_dims(data[:, :, :, 80], axis=0)  # Add batch dimension

print(img.shape)  # Shape: (1, 4, 240, 240)

p = 24
x = rearrange(img, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = p, p2 = p)

print(x.shape)  # Shape: (1, 240*240, 4*24*24)

# Calculate global min and max values across all patches
all_patches = np.array([x[0, i].reshape(p, p, 4)[:, :, 2] for i in range(100)])
vmin = all_patches.min()
vmax = all_patches.max()

# Define the range of patches to display
start_patch = 32  # Starting index of the patches
end_patch = 38 # Ending index of the patches

# Ensure the range is within bounds
start_patch = max(0, start_patch)
end_patch = min(x.shape[1], end_patch)

# Plot the selected patches in a column
fig, axes = plt.subplots(end_patch - start_patch, 1, figsize=(5, 2 * (end_patch - start_patch)))
for idx, patch_idx in enumerate(range(start_patch, end_patch)):
    patch = x[0, patch_idx].reshape(p, p, 4)[:, :, 3]  # Reshape and select the desired channel
    axes[idx].imshow(patch, cmap='gray', vmin=vmin, vmax=vmax)
    axes[idx].axis('off')
    # [idx].set_title(f'Patch {patch_idx}')

plt.tight_layout()
plt.subplots_adjust(wspace=0.01, hspace=0.01)  # smaller = thinner grid
plt.show()